## Reddit thread scraper with Frontier LLMs in Python

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [3]:
# scraper utilities
from bs4 import BeautifulSoup
import requests


# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]


In [11]:
# set up environment

MODEL_LLAMA = 'llama3.2:1b'
OLLAMA_BASE_URL = "http://127.0.0.1:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
url = "https://www.reddit.com/r/mbti/comments/1pzp59z/is_chatgpt_informative_on_mbti_theory_mbti/"

In [ ]:
content = fetch_website_contents(url)
content
links = fetch_website_links(url)
links

In [ ]:
print(content)

In [15]:
# set up prompts

system_prompt = """
You are an assistant that analyzes the contents of a website,
and provides a short summary.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to the post that is sent on the given page, so you can include only the post and the comments made to it.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "post page", "url": "https://full.url/goes/here/about"},
    ]
}
"""

In [16]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [17]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [18]:
def get_links_user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative links):

    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [19]:
user_prompt = get_links_user_prompt(url)

In [22]:
def create_summary(url):
    stream = ollama.chat.completions.create(
        model= MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt[:5000] + fetch_page_and_all_relevant_links(url)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response),display_id=display_handle.display_id)
    

In [23]:
create_summary("https://www.reddit.com/r/mbti/comments/1pzp59z/is_chatgpt_informative_on_mbti_theory_mbti/")

InternalServerError: Error code: 500 - {'error': {'message': 'llama runner process has terminated: error loading model: unable to allocate CPU buffer', 'type': 'api_error', 'param': None, 'code': None}}